# Convert Ontology to Natural Language

This notebook converts the axioms in `odfa.owl` into natural-language-like statements. For example, the axiom

```
UBERON_0001091 BFO_0000051 some UBERON_0001751
```

becomes

```
calcareous tooth has part dentin
```

The quantifiers (`some`, `only`) are dropped.

We use the [Ontology Access Kit (oaklib)](https://incatools.github.io/ontology-access-kit/) Python library.

## Step 1: Convert `odfa.owl` into an `odfa.db` SQLite database

oaklib reads ontologies through *adapters*. The [SQLite adapter](https://incatools.github.io/ontology-access-kit/packages/implementations/sqldb.html) is the most capable and fastest one. It reads a database in the [Semantic-SQL](https://github.com/INCATools/semantic-sql) format.

If you give oaklib a locator of the form `sqlite:path/to/file.owl`, it calls the `semsql` builder to make `path/to/file.db` next to the OWL file and then opens it. The builder runs a Makefile and only rebuilds the `.db` when the `.owl` file is newer.

**Requirements:** the builder needs these command-line tools on your `PATH`:
- [`rdftab`](https://github.com/ontodev/rdftab.rs) loads the RDF triples into SQLite
- [`relation-graph`](https://github.com/INCATools/relation-graph) computes inferred (entailed) edges
- [`robot`](https://robot.obolibrary.org/) strips axioms that are not needed before `relation-graph` runs
- `riot` (from Apache Jena) is used during preprocessing

The build also leaves behind `odfa-relation-graph.tsv.gz`, the inferred edges that were loaded into the database. You can delete it safely.

Instead of installing these, you can run the notebook inside the [ODK docker image](https://github.com/INCATools/ontology-development-kit), which already has them.

In [1]:
import logging
import shutil
from pathlib import Path

import pandas as pd

from oaklib import get_adapter
from oaklib.implementations.sqldb.sql_implementation import SqlImplementation

# Only show warnings and errors. Change to logging.INFO to see every SQL query oaklib runs (very verbose).
logging.basicConfig(level=logging.WARNING)

OWL_FILE = Path("odfa.owl")
DB_FILE = OWL_FILE.with_suffix(".db")  # odfa.db

/opt/homebrew/lib/python3.11/site-packages/eutils/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


---

## helper functions

In [2]:
def check_build_tools(tools=("robot", "rdftab", "relation-graph", "riot")) -> bool:
    """
    Check that the command-line tools needed to build a Semantic-SQL database are installed.

    Building a .db file from a .owl file runs external programs (not Python
    packages). This function looks for each one on your PATH, the list of
    directories your shell searches for programs.

    Args:
        tools: Names of the programs to look for.

    Returns:
        True if every tool was found, otherwise False. Missing tools are
        printed so you know what to install.
    """
    missing = [tool for tool in tools if shutil.which(tool) is None]
    for tool in missing:
        print(f"Missing required tool: {tool}")
    return not missing

In [3]:
def build_sqlite_db(owl_path: Path, force: bool = False) -> Path:
    """
    Convert an OWL ontology file into a Semantic-SQL SQLite database.

    The SQLite file is written next to the OWL file and uses the same name
    with a .db extension (odfa.owl -> odfa.db).

    How it works: when get_adapter() gets a locator such as "sqlite:odfa.owl",
    oaklib sees the .owl extension and calls the semsql builder to make the
    .db file. If the .db file already exists and is newer than the .owl file,
    oaklib reuses it.

    Args:
        owl_path: Path to the .owl file (e.g., Path("odfa.owl")).
        force: If True, delete any existing .db file first so it is rebuilt
            from scratch.

    Returns:
        Path to the SQLite database file.
    """
    owl_path = Path(owl_path)
    if not owl_path.exists():
        raise FileNotFoundError(f"OWL file not found: {owl_path}")

    db_path = owl_path.with_suffix(".db")
    if force and db_path.exists():
        db_path.unlink()

    # Getting an adapter with a "sqlite:<file>.owl" locator triggers the build.
    # We don't need the adapter here; we only want the .db file it creates.
    get_adapter(f"sqlite:{owl_path}")

    if not db_path.exists():
        raise RuntimeError(f"Expected {db_path} to be created, but it was not.")
    return db_path

In [4]:
def load_ontology(db_path: Path) -> SqlImplementation:
    """
    Open a Semantic-SQL SQLite database with oaklib.

    The returned adapter is the object you use to query the ontology, for
    example adapter.label("UBERON:0001091") or adapter.relationships(...).

    Args:
        db_path: Path to the .db file made by build_sqlite_db().

    Returns:
        An oaklib SqlImplementation adapter connected to the database.
    """
    db_path = Path(db_path)
    if not db_path.exists():
        raise FileNotFoundError(f"Database not found: {db_path}. Run build_sqlite_db() first.")
    return get_adapter(f"sqlite:{db_path}")

In [5]:
def to_obo_curie(curie: str) -> str:
    """
    Convert a CURIE into the "obo:PREFIX_ID" form, e.g. "EMRO:0000000" -> "obo:EMRO_0000000".

    Why this is needed: semsql shortens IRIs using its own prefix table. Some
    prefixes in odfa.owl (e.g., EMRO, UBPROP) are missing from that table, so
    the database stores http://purl.obolibrary.org/obo/EMRO_0000000 as
    "obo:EMRO_0000000". relation-graph shortens the same IRI to
    "EMRO:0000000". To look up these terms in the database, we convert to the
    form the database uses.

    Args:
        curie: A CURIE such as "EMRO:0000000".

    Returns:
        The same identifier written as "obo:PREFIX_ID".
    """
    prefix, local_id = curie.split(":", 1)
    return f"obo:{prefix}_{local_id}"

In [6]:
def from_obo_curie(curie: str) -> str:
    """
    Convert an "obo:PREFIX_ID" CURIE back to "PREFIX:ID", e.g. "obo:EMRO_0000000" -> "EMRO:0000000".

    This is the reverse of to_obo_curie(). oaklib returns identifiers the way
    the database stores them ("obo:EMRO_0000000"), but the relation graph uses
    "EMRO:0000000". Converting to the relation-graph form lets us compare the two.

    Args:
        curie: Any CURIE. Values that are not in the "obo:PREFIX_ID" form are
            returned unchanged.

    Returns:
        The CURIE in "PREFIX:ID" form.
    """
    if curie.startswith("obo:") and "_" in curie:
        prefix, local_id = curie[len("obo:"):].split("_", 1)
        return f"{prefix}:{local_id}"
    return curie

In [7]:
def get_one_label_per_curie(adapter: SqlImplementation, curies) -> dict:
    """
    Look up labels for many CURIEs at once, returning exactly one label per CURIE.

    Calling adapter.label() once per CURIE runs one database query each time.
    adapter.labels() takes many CURIEs and looks them up together, which is
    much faster for a large table. However, adapter.labels() returns every
    label a term has, and some terms have more than one. For example,
    BFO:0000051 has the labels "has part" and "has_part", because the
    ontologies imported into ODFA don't all agree.

    For CURIEs with more than one label, we call adapter.label(), which
    returns a single label (the first one stored in the database). That keeps
    our labels consistent with what adapter.label() shows elsewhere.

    Args:
        adapter: An oaklib adapter, e.g. the one returned by load_ontology().
        curies: Any iterable of CURIEs.

    Returns:
        A dict such as {"BFO:0000051": "has part", ...}. CURIEs without a
        label are left out of the dict.
    """
    # Collect all labels for each CURIE, e.g. {"BFO:0000051": ["has part", "has part", "has_part"]}.
    # adapter.labels() yields (curie, label) pairs; the label is None if there isn't one.
    all_labels = {}
    for curie, label in adapter.labels(list(curies)):
        if label:
            all_labels.setdefault(curie, []).append(label)

    # Keep the only label, or let adapter.label() pick one when the labels differ.
    return {
        curie: labels[0] if len(set(labels)) == 1 else adapter.label(curie)
        for curie, labels in all_labels.items()
    }

In [8]:
# Labels for built-in predicates that have no rdfs:label in the ontology.
CUSTOM_LABELS = {
    "rdfs:subClassOf": "is a",
    # OBI:0000417 is labelled "achieves_planned_objective" in the ontology; write it with spaces.
    "OBI:0000417": "achieves planned objective",
}


def get_label_map(adapter: SqlImplementation, curies, custom_labels: dict = CUSTOM_LABELS) -> dict:
    """
    Build a dictionary that maps each CURIE to its label.

    Labels are looked up in one batch with get_one_label_per_curie(), which
    also makes sure each CURIE gets exactly one label.

    If a CURIE has no label, we retry with its "obo:" form (see to_obo_curie()),
    because some prefixes are stored that way in the database.

    Built-in predicates such as rdfs:subClassOf have no label in the ontology,
    so we fill them in from custom_labels (e.g., rdfs:subClassOf -> "is a").

    Args:
        adapter: An oaklib adapter, e.g. the one returned by load_ontology().
        curies: Any iterable of CURIEs, e.g. "UBERON:0001091". Duplicates are fine.
        custom_labels: A dict of {CURIE: label} applied after the ontology
            lookup. These override ontology labels. Defaults to CUSTOM_LABELS.

    Returns:
        A dict such as {"UBERON:0001091": "calcareous tooth", ...}, keyed by
        the CURIEs as given. CURIEs without a label are left out of the dict.
    """
    unique_curies = list(set(curies))
    label_map = get_one_label_per_curie(adapter, unique_curies)

    # Retry unlabeled CURIEs in the "obo:PREFIX_ID" form.
    # obo_form maps the obo form back to the original CURIE, e.g. {"obo:EMRO_0000000": "EMRO:0000000"}.
    obo_form = {to_obo_curie(curie): curie for curie in unique_curies if curie not in label_map and ":" in curie}
    for obo_curie, label in get_one_label_per_curie(adapter, obo_form).items():
        label_map[obo_form[obo_curie]] = label

    # Add custom labels for CURIEs that are in the input.
    label_map.update({curie: label for curie, label in custom_labels.items() if curie in unique_curies})
    return label_map

In [9]:
def get_asserted_edges(adapter: SqlImplementation) -> pd.DataFrame:
    """
    Get the edges that are stated directly in the ontology (asserted), not inferred.

    adapter.relationships() returns asserted edges by default
    (include_entailed=False). These include simple subclass axioms such as
    "UBERON:0001091 subClassOf UBERON:0010314" and existential restrictions
    such as "UBERON:0001091 has part some UBERON:0001751", which oaklib
    returns as the triple (UBERON:0001091, BFO:0000051, UBERON:0001751).

    Args:
        adapter: An oaklib adapter, e.g. the one returned by load_ontology().

    Returns:
        A DataFrame with "subject", "predicate", and "object" columns. The
        CURIEs are converted with from_obo_curie() so they match the
        relation graph.
    """
    # Each relationship is a (subject, predicate, object) tuple.
    edges = [tuple(from_obo_curie(curie) for curie in edge) for edge in adapter.relationships()]
    return pd.DataFrame(edges, columns=["subject", "predicate", "object"]).drop_duplicates()

In [10]:
def add_asserted_column(df: pd.DataFrame, adapter: SqlImplementation) -> pd.DataFrame:
    """
    Add an "asserted" column: True if the edge is stated in the ontology, False if it was inferred.

    The relation graph mixes asserted edges with edges that relation-graph
    inferred. For example, "calcareous tooth is a tooth-like structure" is
    asserted, while "calcareous tooth is a continuant" is inferred by following
    "is a" links up the hierarchy.

    relation-graph doesn't output every asserted edge (in ODFA it misses
    "psychophysiological response to emotion causally related to emotion
    process"). So any asserted edge whose subject is already in df but which is
    missing from df is added at the end, with asserted=True. Asserted edges about
    properties (e.g., rdfs:domain) are not added, because their subjects are
    not in df.

    Args:
        df: A DataFrame with "subject", "predicate", and "object" columns. It is not modified.
        adapter: An oaklib adapter used to get the asserted edges.

    Returns:
        A copy of df with an "asserted" column of True/False values, plus any
        missing asserted edges.
    """
    columns = ["subject", "predicate", "object"]
    asserted_df = get_asserted_edges(adapter)
    # Only keep asserted edges about subjects that are in df (i.e., classes).
    asserted_df = asserted_df[asserted_df["subject"].isin(df["subject"])]

    # A left merge keeps every row of df in its original order. indicator=True adds a "_merge"
    # column that says "both" when the row was also found in asserted_df.
    merged = df.merge(asserted_df, on=columns, how="left", indicator=True)
    merged["asserted"] = merged["_merge"] == "both"
    merged = merged.drop(columns="_merge")

    # Find asserted edges that are not in df at all, and add them at the end.
    missing = asserted_df.merge(df[columns], on=columns, how="left", indicator=True)
    missing = missing[missing["_merge"] == "left_only"].drop(columns="_merge")
    missing["asserted"] = True

    return pd.concat([merged, missing], ignore_index=True)

In [11]:
def add_statement_column(df: pd.DataFrame, adapter: SqlImplementation) -> pd.DataFrame:
    """
    Add a "statement" column that turns each subject/predicate/object row into a sentence.

    For example, the row
        subject=UBERON:0001091  predicate=BFO:0000051  object=UBERON:0001751
    gets the statement "calcareous tooth has part dentine".

    The subject, predicate, and object columns keep their CURIEs, so each
    statement can still be traced back to the terms it came from. Any CURIE
    without a label is used as it is in the statement.

    Args:
        df: A DataFrame with "subject", "predicate", and "object" columns of CURIEs.
            It is not modified.
        adapter: An oaklib adapter used to look up labels.

    Returns:
        A copy of df with a new "statement" column.
    """
    columns = ["subject", "predicate", "object"]

    # Collect every CURIE from the three columns so all labels are looked up in one batch.
    label_map = get_label_map(adapter, pd.unique(df[columns].values.ravel()))

    statements_df = df.copy()
    # For each column, swap CURIEs for labels. .map() returns NaN for CURIEs missing
    # from label_map; .fillna() puts the original CURIE back in those places.
    labels = {column: df[column].map(label_map).fillna(df[column]) for column in columns}
    statements_df["statement"] = labels["subject"] + " " + labels["predicate"] + " " + labels["object"]
    return statements_df

In [12]:
def write_statements_file(
    statements_df: pd.DataFrame,
    output_file: Path = Path("odfa-natural-language.txt"),
    asserted_only: bool = True,
) -> Path:
    """
    Write the natural language statements to a text file, one statement per line.

    Each line ends with a period, e.g. "calcareous tooth has part dentine.".
    There is no header row and no index column. We write the lines ourselves
    because statements_df.to_csv() would put quotes around statements that
    contain spaces.

    Args:
        statements_df: A DataFrame from add_statement_column(), with
            "statement" and "asserted" columns.
        output_file: Where to write the file. Defaults to odfa-natural-language.txt.
        asserted_only: If True (the default), write only the statements stated
            directly in the ontology (asserted == True). If False, also write
            the statements relation-graph inferred.

    Returns:
        The path of the file that was written.
    """
    output_file = Path(output_file)

    # Keep only the asserted rows if asked. statements_df["asserted"] is a column of
    # True/False values, so using it inside [...] keeps just the rows marked True.
    if asserted_only:
        statements_df = statements_df[statements_df["asserted"]]

    output_file.write_text("".join(statement + ".\n" for statement in statements_df["statement"]), encoding="utf-8")
    return output_file

In [13]:
def get_stanza_map(adapter: SqlImplementation, curies) -> dict:
    """
    Map each CURIE to the stanza(s) its axioms are stored under in the database.

    What is a stanza? In a Semantic-SQL database, every row of the statements
    table has a "stanza" column holding the class the row belongs to. For a
    simple axiom like "UBERON:0001091 subClassOf UBERON:0010314", the stanza
    is the subject, UBERON:0001091. For an axiom like "UBERON:0001091 has part
    some UBERON:0001751", OWL stores the "has part some dentine" part as a
    blank node (an unnamed node such as "_:riog00001070"). Those rows have the
    blank node as their subject, but their stanza is still UBERON:0001091.
    The stanza therefore groups all of a class's axioms together.

    oaklib's database model doesn't include the stanza column, so we read it
    with SQL through the adapter's database connection (adapter.engine).

    Args:
        adapter: An oaklib SqlImplementation adapter.
        curies: Any iterable of subject CURIEs, e.g. "UBERON:0001091".

    Returns:
        A dict such as {"UBERON:0001091": ["UBERON:0001091"], ...}, keyed by the
        CURIEs as given. Each value is a list because a subject could in
        principle appear in more than one stanza. CURIEs with no stanza are
        left out of the dict.
    """
    # Read every (subject, stanza) pair for named subjects. Blank nodes start with "_:" and are skipped.
    pairs_df = pd.read_sql(
        "SELECT DISTINCT subject, stanza FROM statements WHERE subject NOT LIKE '\\_:%' ESCAPE '\\'",
        adapter.engine,
    )
    # Group the pairs into {subject: [stanza, ...]}.
    db_stanzas = pairs_df.groupby("subject")["stanza"].apply(list).to_dict()

    stanza_map = {}
    for curie in set(curies):
        # Try the CURIE as given, then its "obo:" form (see to_obo_curie()).
        stanzas = db_stanzas.get(curie) or (db_stanzas.get(to_obo_curie(curie)) if ":" in curie else None)
        if stanzas:
            stanza_map[curie] = stanzas
    return stanza_map

In [14]:
def build_iri_statement_df(statements_df: pd.DataFrame, adapter: SqlImplementation) -> pd.DataFrame:
    """
    Build a DataFrame that maps each class IRI to its natural language statements.

    Each statement is assigned to the stanza of its subject (see
    get_stanza_map()). The stanza is turned into a full IRI, e.g.
    "UBERON:0001091" -> "http://purl.obolibrary.org/obo/UBERON_0001091", so
    every IRI is written the same way, including the "obo:EMRO_..." ones.

    The mapping can be many-to-many: an IRI usually has several statements,
    and the same statement text can belong to more than one IRI (for example,
    if two classes share a label, or a subject appears in several stanzas).

    Args:
        statements_df: A DataFrame from add_statement_column(), with "subject",
            "statement", and "asserted" columns.
        adapter: An oaklib adapter used to find stanzas and expand CURIEs to IRIs.

    Returns:
        A DataFrame with "iri", "statement", and "asserted" columns, one row
        per (IRI, statement) pair, without duplicates. Subjects with no stanza
        are left out.
    """
    stanza_map = get_stanza_map(adapter, statements_df["subject"])

    # Give each row a list of stanzas, then explode() so a row with two stanzas becomes two rows.
    iri_df = statements_df[["subject", "statement", "asserted"]].copy()
    iri_df["stanza"] = iri_df["subject"].map(stanza_map)
    iri_df = iri_df.dropna(subset=["stanza"]).explode("stanza")

    # Expand each stanza CURIE into a full IRI. curie_to_uri() returns None if the prefix is unknown,
    # in which case we keep the stanza as it is.
    iri_map = {stanza: adapter.curie_to_uri(stanza) or stanza for stanza in iri_df["stanza"].unique()}
    iri_df["iri"] = iri_df["stanza"].map(iri_map)

    return iri_df[["iri", "statement", "asserted"]].drop_duplicates().reset_index(drop=True)

In [15]:
def group_statements_by_iri(iri_statement_df: pd.DataFrame, separator: str = ". ") -> pd.DataFrame:
    """
    Combine all of an IRI's statements into one row.

    iri_statement_df has one row per (IRI, statement) pair, so an IRI with 36
    statements takes up 36 rows. This function returns one row per IRI with
    two text columns:

    - "asserted_statements": only the statements stated directly in the ontology.
    - "statements": all statements, asserted and inferred.

    The statements are sorted alphabetically within each IRI, so the output is
    the same every time the notebook runs. They are joined with separator, and
    a final "." ends the last statement, e.g.
    "calcareous tooth has part dentine. calcareous tooth is a tooth-like structure."

    Args:
        iri_statement_df: A DataFrame from build_iri_statement_df(), with
            "iri", "statement", and "asserted" columns.
        separator: The text placed between statements. Defaults to ". ".

    Returns:
        A DataFrame with "iri", "asserted_statements", and "statements"
        columns, one row per IRI, sorted by IRI. An IRI with no asserted
        statements has an empty string in "asserted_statements".
    """

    def join_statements(statements: pd.Series) -> str:
        # Join the statements and end the last one with a period.
        return separator.join(statements) + "."

    # sort_values() orders the statements before they are joined.
    sorted_df = iri_statement_df.sort_values(["iri", "statement"])

    # groupby("iri") puts each IRI's rows together; agg(join_statements) joins their statements into one string.
    all_statements = sorted_df.groupby("iri")["statement"].agg(join_statements)
    asserted_statements = sorted_df[sorted_df["asserted"]].groupby("iri")["statement"].agg(join_statements)

    # reindex() lines the asserted statements up with every IRI, using "" for IRIs that have none.
    return pd.DataFrame(
        {
            "asserted_statements": asserted_statements.reindex(all_statements.index, fill_value=""),
            "statements": all_statements,
        }
    ).reset_index()

---

## convert ontology into sqlite database

In [16]:
# Build odfa.db (this takes a few seconds the first time) and open it.
if check_build_tools():
    db_path = build_sqlite_db(OWL_FILE)
    print(f"Database ready: {db_path} ({db_path.stat().st_size / 1e6:.1f} MB)")

adapter = load_ontology(DB_FILE)

make: `odfa.db' is up to date.
Database ready: odfa.db (1.8 MB)


### Sanity check

Look up the labels in the example axiom and list the relationships for *calcareous tooth* (`UBERON:0001091`). oaklib writes identifiers as CURIEs (`UBERON:0001091`) rather than the `UBERON_0001091` form used in OWL IRIs.

In [17]:
print("Number of entities:", len(list(adapter.entities())))

for curie in ["UBERON:0001091", "BFO:0000051", "UBERON:0001751"]:
    print(f"{curie} -> {adapter.label(curie)}")

print()
for subject, predicate, obj in adapter.relationships(["UBERON:0001091"]):
    print(subject, predicate, obj)

Number of entities: 350
UBERON:0001091 -> calcareous tooth
BFO:0000051 -> has part
UBERON:0001751 -> dentine

UBERON:0001091 BFO:0000051 UBERON:0001751
UBERON:0001091 rdfs:subClassOf OHD:0000206
UBERON:0001091 rdfs:subClassOf UBERON:0003913
UBERON:0001091 rdfs:subClassOf UBERON:0010314


---

## Inspect the relation graph

When the database is built, `relation-graph` writes `odfa-relation-graph.tsv.gz`, which contains every edge it inferred (entailed), including reflexive `rdfs:subClassOf` edges such as `X subClassOf X`. These edges are loaded into the database's `entailed_edge` table. The file is tab-separated and has no header row, so we pass column names to pandas. pandas decompresses the `.gz` file automatically.

In [18]:
relation_graph_df = pd.read_csv(
    "odfa-relation-graph.tsv.gz",
    sep="\t",
    header=None,
    names=["subject", "predicate", "object"],
)
print(f"{len(relation_graph_df)} rows")
relation_graph_df.head()

2861 rows


,subject,predicate,object
0,OHD:0000430,rdfs:subClassOf,OHD:0000430
1,OHD:0000446,rdfs:subClassOf,OHD:0000445
2,OHD:0000449,rdfs:subClassOf,OHD:0000445
3,OHD:0000445,rdfs:subClassOf,OHD:0000445
4,OHD:0000448,rdfs:subClassOf,OHD:0000445


### Drop reflexive edges

relation-graph lists every class as a subclass of itself (`X subClassOf X`). Sentences like "X is a X" carry no information, so we keep only rows where the subject and object differ.

In [19]:
relation_graph_df = relation_graph_df[relation_graph_df.subject != relation_graph_df.object]
print(f"{len(relation_graph_df)} rows after dropping reflexive edges")
relation_graph_df.head()

2602 rows after dropping reflexive edges


,subject,predicate,object
1,OHD:0000446,rdfs:subClassOf,OHD:0000445
2,OHD:0000449,rdfs:subClassOf,OHD:0000445
4,OHD:0000448,rdfs:subClassOf,OHD:0000445
5,OHD:0000450,rdfs:subClassOf,OHD:0000445
6,OHD:0000447,rdfs:subClassOf,OHD:0000445


### Mark asserted edges

Most relation-graph edges are inferred. `add_asserted_column()` uses oaklib's `adapter.relationships()`, which returns only the edges stated directly in the ontology, to add an `asserted` column (True/False). It also adds the one asserted class axiom that relation-graph left out.

In [20]:
relation_graph_df = add_asserted_column(relation_graph_df, adapter)
print(f"{len(relation_graph_df)} rows, {relation_graph_df['asserted'].sum()} asserted")
relation_graph_df.head()

2603 rows, 383 asserted


,subject,predicate,object,asserted
0,OHD:0000446,rdfs:subClassOf,OHD:0000445,True
1,OHD:0000449,rdfs:subClassOf,OHD:0000445,True
2,OHD:0000448,rdfs:subClassOf,OHD:0000445,True
3,OHD:0000450,rdfs:subClassOf,OHD:0000445,True
4,OHD:0000447,rdfs:subClassOf,OHD:0000445,True


---

## Create natural language statements

The columns hold CURIEs (short forms of IRIs, e.g. `UBERON:0001091` stands for `http://purl.obolibrary.org/obo/UBERON_0001091`). `add_statement_column()` uses oaklib to look up each CURIE's `rdfs:label` and joins the labels into a `statement` column, e.g. "calcareous tooth has part dentine". The `subject`, `predicate`, and `object` columns keep their CURIEs, so each statement can be traced back to its terms.

Built-in predicates have no label in the ontology, so we supply our own in `CUSTOM_LABELS` (`rdfs:subClassOf` becomes "is a"). Any other CURIE without a label is used as it is.

In [21]:
statements_df = add_statement_column(relation_graph_df, adapter)
statements_df.head()

,subject,predicate,object,asserted,statement
0,OHD:0000446,rdfs:subClassOf,OHD:0000445,True,MDAS 1. If you went to your Dentist for TREATM...
1,OHD:0000449,rdfs:subClassOf,OHD:0000445,True,MDAS 4. If you were about to have your TEETH S...
2,OHD:0000448,rdfs:subClassOf,OHD:0000445,True,MDAS 3. If you were about to have a TOOTH DRIL...
3,OHD:0000450,rdfs:subClassOf,OHD:0000445,True,MDAS 5. If you were about to have a LOCAL ANAE...
4,OHD:0000447,rdfs:subClassOf,OHD:0000445,True,MDAS 2. If you were sitting in the WAITING ROO...


In [22]:
statements_df.tail()

,subject,predicate,object,asserted,statement
2598,OHD:0008067,IAO:0000136,EMRO:0003001,False,IDAF-4C+ 2e. I am afraid of going to the denti...
2599,OHD:0008069,IAO:0000136,EMRO:0003002,True,IDAF-4C+ 3b. Feeling embarrassed or ashamed. i...
2600,OHD:0008065,IAO:0000136,EMRO:0003002,True,"IDAF-4C+ 2c. My fear, anxiety or avoidance of ..."
2601,OHD:0008067,IAO:0000136,EMRO:0003002,True,IDAF-4C+ 2e. I am afraid of going to the denti...
2602,EMRO:0000007,RO:0002410,EMRO:0000000,True,psychophysiological response to emotion causal...


---

## Save output

`write_statements_file()` writes the statements to `odfa-natural-language.txt`, one per line, each ending with a period. By default (`asserted_only=True`) only the statements stated directly in the ontology are written. Pass `asserted_only=False` to also include the statements relation-graph inferred.

In [23]:
output_file = write_statements_file(statements_df, asserted_only=True)

# Read the file back and show its first and last five lines.
file_lines = output_file.read_text(encoding="utf-8").splitlines()
print(f"Wrote {len(file_lines)} lines to {output_file}\n")
print("First five lines:")
print("\n".join(file_lines[:5]))
print("\nLast five lines:")
print("\n".join(file_lines[-5:]))

Wrote 383 lines to odfa-natural-language.txt

First five lines:
MDAS 1. If you went to your Dentist for TREATMENT TOMORROW, how would you feel? is a Modified Dental Anxiety Scale (MDAS) question.
MDAS 4. If you were about to have your TEETH SCALED AND POLISHED, how would you feel? is a Modified Dental Anxiety Scale (MDAS) question.
MDAS 3. If you were about to have a TOOTH DRILLED, how would you feel? is a Modified Dental Anxiety Scale (MDAS) question.
MDAS 5. If you were about to have a LOCAL ANAESTHETIC INJECTION in your gum, above an upper back tooth, how would you feel? is a Modified Dental Anxiety Scale (MDAS) question.
MDAS 2. If you were sitting in the WAITING ROOM (waiting for treatment), how would you feel? is a Modified Dental Anxiety Scale (MDAS) question.

Last five lines:
IDAF-4C+ 1g. I delay making appointments to go to the dentist. is about dental visit avoidance behavior that involves delay in scheduling a dental appointment.
IDAF-4C+ 3b. Feeling embarrassed or ashamed.

---

## Associate stanzas with natural language statements

In `odfa.db`, each row of the `statements` table has a `stanza` column: the class the axiom belongs to. Grouping statements by stanza gives the set of natural language statements for a particular IRI.

Each row of `statements_df` is assigned to the stanza of its subject. In this database the stanza of a named subject is always the subject itself. Blank nodes (e.g. the `has part some dentine` part of an axiom) take the stanza of the class that uses them. This also gives inferred edges, which aren't stored as axioms in any stanza, a stanza: the one of their subject.

The result `iri_statement_df` maps each IRI to its statements. The mapping can be many-to-many: one IRI has many statements, and one statement can belong to more than one IRI.

### Asserted statements vs. all statements

When the statements are grouped by IRI (`iri_statements_df`, below), each IRI gets two columns:

- **`asserted_statements`**: only the axioms written in the ontology for that class, i.e. what an ontology editor stated directly. For *calcareous tooth* these are "has part dentine" and its 3 direct superclasses (*functional tooth*, *tooth-like structure*, and *structure with developmental contribution from neural crest*). An IRI with no asserted axioms of its own would get an empty string, though every IRI in the current ODFA has at least one.
- **`statements`**: the asserted statements **plus** everything relation-graph inferred from them. Inferred statements follow "is a" links up the hierarchy (*calcareous tooth is a tooth-like structure*, so it is also *a skeletal element*, *a material entity*, and *a continuant*) and carry relations along them (*has part dentine* implies *has part tissue*). They are logically correct but often very general.

Use `asserted_statements` for a short, curated description of each class, and `statements` when you want everything the ontology implies. Neither column includes `only` restrictions or axioms that use `or`, because these don't appear in relation-graph or `adapter.relationships()` output.

In [24]:
iri_statement_df = build_iri_statement_df(statements_df, adapter)
print(f"{len(iri_statement_df)} (IRI, statement) pairs for {iri_statement_df['iri'].nunique()} IRIs")
iri_statement_df.head()

2603 (IRI, statement) pairs for 254 IRIs


,iri,statement,asserted
0,http://purl.obolibrary.org/obo/OHD_0000446,MDAS 1. If you went to your Dentist for TREATM...,True
1,http://purl.obolibrary.org/obo/OHD_0000449,MDAS 4. If you were about to have your TEETH S...,True
2,http://purl.obolibrary.org/obo/OHD_0000448,MDAS 3. If you were about to have a TOOTH DRIL...,True
3,http://purl.obolibrary.org/obo/OHD_0000450,MDAS 5. If you were about to have a LOCAL ANAE...,True
4,http://purl.obolibrary.org/obo/OHD_0000447,MDAS 2. If you were sitting in the WAITING ROO...,True


### Statements for one IRI

The statements for *calcareous tooth* (`UBERON:0001091`). They include inferred statements such as superclasses of its superclasses.

In [25]:
iri = adapter.curie_to_uri("UBERON:0001091")
iri_statement_df[iri_statement_df["iri"] == iri].head()

,iri,statement,asserted
543,http://purl.obolibrary.org/obo/UBERON_0001091,calcareous tooth is a material anatomical entity,False
560,http://purl.obolibrary.org/obo/UBERON_0001091,calcareous tooth is a material entity,False
621,http://purl.obolibrary.org/obo/UBERON_0001091,calcareous tooth is a continuant,False
1067,http://purl.obolibrary.org/obo/UBERON_0001091,calcareous tooth is a structure with developme...,True
1240,http://purl.obolibrary.org/obo/UBERON_0001091,calcareous tooth is a skeletal element,False


### Check for statements that belong to more than one IRI

In [26]:
iri_counts = iri_statement_df.groupby("statement")["iri"].nunique()
shared = iri_counts[iri_counts > 1]
print(f"{len(shared)} statements belong to more than one IRI")
iri_statement_df[iri_statement_df["statement"].isin(shared.index)].sort_values("statement")

0 statements belong to more than one IRI


,iri,statement,asserted


### Group multiple statements to a single IRI

In [27]:
iri_statements_df = group_statements_by_iri(iri_statement_df)
print(f"{len(iri_statements_df)} IRIs")
iri_statements_df.head()

254 IRIs


,iri,asserted_statements,statements
0,http://purl.obolibrary.org/obo/BFO_0000004,independent continuant is a continuant.,independent continuant is a continuant.
1,http://purl.obolibrary.org/obo/BFO_0000015,process is a occurrent.,process is a occurrent.
2,http://purl.obolibrary.org/obo/BFO_0000016,disposition is a realizable entity.,disposition is a continuant. disposition is a ...
3,http://purl.obolibrary.org/obo/BFO_0000017,realizable entity is a specifically dependent ...,realizable entity is a continuant. realizable ...
4,http://purl.obolibrary.org/obo/BFO_0000020,specifically dependent continuant is a continu...,specifically dependent continuant is a continu...


In [28]:
# Print the grouped statements for calcareous tooth (UBERON:0001091).
iri = adapter.curie_to_uri("UBERON:0001091")
row = iri_statements_df[iri_statements_df["iri"] == iri].iloc[0]
print(iri)
print("\nasserted_statements:")
print(row["asserted_statements"])
print("\nstatements:")
print(row["statements"])

http://purl.obolibrary.org/obo/UBERON_0001091

asserted_statements:
calcareous tooth has part dentine. calcareous tooth is a functional tooth. calcareous tooth is a structure with developmental contribution from neural crest. calcareous tooth is a tooth-like structure.

statements:
calcareous tooth has part anatomical entity. calcareous tooth has part anatomical structure. calcareous tooth has part connective tissue. calcareous tooth has part continuant. calcareous tooth has part dentine. calcareous tooth has part ectoderm-derived structure. calcareous tooth has part independent continuant. calcareous tooth has part material anatomical entity. calcareous tooth has part material entity. calcareous tooth has part mineralized skeletal tissue. calcareous tooth has part multicellular anatomical structure. calcareous tooth has part neural crest-derived structure. calcareous tooth has part odontoid tissue. calcareous tooth has part skeletal tissue. calcareous tooth has part structure with dev